# SIC Survival Head

- **SIC** is Survival Head Independent. By default, we use the DeepHit10 (discrete-time head with 10 bins)
- Additionally we provide the DeepHit50 / DeepHit100 (different discretization) and continouos-time DeepEH (extended hazard) and CoxPH (DeepSurv)

- Note that the provided chackpoints are final checkpoints after 2 stages and may differ from the reported in the paper (after stage 1 or stage 1 and 2000 steps)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sic import SIC, list_datasets, SURVIVAL_HEADS
from sic.metrics import cross_validate, summarize

RANDOM_STATE = 42
K_FOLDS = 5
CSV_PATH = "head_comparison_results.csv"

HEADS = ["DeepHit10", "DeepHit50", "DeepHit100", "CoxPH", "DeepEH"]
DATASETS = list_datasets()
print(f"{len(HEADS)} heads x {len(DATASETS)} datasets x {K_FOLDS} folds")

## Run cross-validation for every head

This is the heavy step: it loops heads x datasets x folds. Each head loads its own checkpoint from the Hub on first use. Results from every head are concatenated (with a `head` column) and written to one CSV so the plots can be regenerated without re-running.

In [ ]:
# FLCHAIN by DeepEH is a problem and outputs nans
all_rows = []
for head in HEADS:
    print(f"\n=== {head} ===")
    model = SIC(survival_head=head, standardize=False, random_state=RANDOM_STATE)
    df_head = cross_validate(model, DATASETS, k_folds=K_FOLDS, seed=RANDOM_STATE)
    df_head.insert(0, "head", head)
    all_rows.append(df_head)

results = pd.concat(all_rows, ignore_index=True)
results.to_csv(CSV_PATH, index=False)
print(f"\nsaved {len(results)} rows to {CSV_PATH}")
results.head()

## Aggregate and order datasets by sample size

We read the CSV back, compute `mean +/- std` per (dataset, head), and fix a single dataset order (smallest to largest sample size) reused by every plot.

In [ ]:
df = pd.read_csv(CSV_PATH)

# Dataset order: smallest -> largest number of samples.
dataset_order = list(df.groupby("dataset")["n_samples"].first().sort_values().index)
n_by_ds = df.groupby("dataset")["n_samples"].first()

HEAD_LABELS = {
    "DeepHit10": "SIC (DH10)",
    "DeepHit50": "SIC (DH50)",
    "DeepHit100": "SIC (DH100)",
    "DeepEH": "SIC (DeepEH)",
    "CoxPH": "SIC (CoxPH)",
}
HIGHLIGHT = "DeepHit10"

def pivot(metric, aggfunc="mean"):
    """(dataset x head) table for a metric, rows ordered by sample size."""
    p = df.pivot_table(index="dataset", columns="head", values=metric, aggfunc=aggfunc)
    return p.reindex(index=dataset_order, columns=HEADS)

cindex_mean = pivot("cindex_td", "mean"); cindex_std = pivot("cindex_td", "std")
ibs_mean = pivot("ibs", "mean"); ibs_std = pivot("ibs", "std")

# Miscalibrated folds: count of folds with D-calibration p-value < 0.05.
ALPHA = 0.05
miscal = (
    df.assign(miscal=(df["d_cal"] < ALPHA).astype(int))
      .pivot_table(index="dataset", columns="head", values="miscal", aggfunc="sum")
      .reindex(index=dataset_order, columns=HEADS)
)
cindex_mean.round(3)

## Dot plots: C-index and IBS

One group of points per dataset (ordered by sample size), one series per head, with std error bars over folds.

In [ ]:
def dot_errorbar(mean_tbl, std_tbl, ylabel, title, hline=None, ylim=None, save_path=None):
    """Dot + error-bar plot per head, per dataset (paper-style)."""
    datasets = list(mean_tbl.index)
    x = np.arange(len(datasets))
    offsets = np.linspace(-0.25, 0.25, len(HEADS))
    xtick_labels = [f"{ds} (n={int(n_by_ds[ds])})" for ds in datasets]

    plt.figure(figsize=(12, 6))
    for i, head in enumerate(HEADS):
        label = HEAD_LABELS[head]
        plt.errorbar(
            x + offsets[i],
            mean_tbl[head].to_numpy(),
            yerr=std_tbl[head].to_numpy(),
            fmt="o",
            capsize=3,
            alpha=0.9 if head == HIGHLIGHT else 0.5,
            label=label,
            zorder=3 if head == HIGHLIGHT else 2,
        )
    if hline is not None:
        plt.axhline(hline, color="k", ls="--", lw=1, zorder=1)
    plt.xticks(x, xtick_labels, rotation=45, ha="right")
    plt.ylabel(ylabel)
    plt.xlabel("Dataset")
    plt.title(title)
    if ylim is not None:
        plt.ylim(*ylim)
    plt.grid(axis="y", alpha=0.3)
    plt.legend(frameon=False, ncol=1, loc="upper left")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=300)
    plt.show()


dot_errorbar(cindex_mean, cindex_std, "Time-dependent C-index",
             "Discrimination by head (higher is better)", hline=0.5, ylim=(0.4, 1.0))

In [ ]:
dot_errorbar(ibs_mean, ibs_std, "IBS",
             "Integrated Brier Score by head (lower is better)")

## Heatmap: miscalibrated folds

Number of folds (out of `K_FOLDS`) where each head is miscalibrated on each dataset (D-calibration p-value < 0.05). Lower / cooler is better; rows ordered by sample size.

In [ ]:
mat = miscal.to_numpy(dtype=float)
fig, ax = plt.subplots(figsize=(1.2 * len(HEADS) + 3, 0.5 * len(dataset_order) + 2))
im = ax.imshow(mat, cmap="Reds", vmin=0, vmax=K_FOLDS, aspect="auto")

ax.set_xticks(range(len(HEADS))); ax.set_xticklabels(HEADS, rotation=30, ha="right")
ax.set_yticks(range(len(dataset_order)))
ax.set_yticklabels([f"{d} (n={int(n_by_ds[d])})" for d in dataset_order])
ax.set_title(f"Miscalibrated folds (p < {ALPHA}), out of {K_FOLDS}")

for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat[i, j]
        txt = "-" if np.isnan(v) else f"{int(v)}"
        ax.text(j, i, txt, ha="center", va="center",
                color="white" if (not np.isnan(v) and v > K_FOLDS / 2) else "black", fontsize=9)

cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("# miscalibrated folds")
plt.tight_layout(); plt.show()